# Patch Generation — GPU (Kaggle)

## Setup steps before running

1. **Enable GPU:** Settings (right panel) → Accelerator → **GPU T4 x2** or P100
2. **Enable Internet:** Settings → Internet → **On** (needed to download model from HuggingFace)
3. **Add test data:** Click **+ Add Input** (top right) → Datasets → Upload → select `test_functions.csv` from your local `data/` folder
4. Edit the CONFIG cell below with the model you want to run
5. **Run All** (Run → Run All)
6. When complete, download the output JSONL from the **Output** tab on the right
   → copy it to your local `results/` folder
7. Run locally: `python validate_patches.py --input results/<output_file>`

**Resuming after a disconnect:** Download the partial JSONL from the Output tab, re-upload it as a new dataset, then re-run — the notebook will detect it and skip completed calls.

In [ ]:
# ── CONFIG — edit this cell before running ──────────────────────────────

# Model options:
#   'Salesforce/codegen-350M-multi'        (Model 1 — C/C++ trained)
#   'deepseek-ai/deepseek-coder-6.7b-base' (Model 2 — best results)
#   'bigcode/starcoder2-3b'                (Model 3 — good fallback)
MODEL_NAME  = 'deepseek-ai/deepseek-coder-6.7b-base'
MODEL_ALIAS = 'deepseek_6.7b'

NUM_SAMPLES    = 5
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.8
MAX_PROMPT_TOKENS = 1600

HF_TOKEN = ''  # only needed for gated models

OUTPUT_FILE = f'/kaggle/working/patches_raw_{MODEL_ALIAS}.jsonl'
print(f'Model   : {MODEL_NAME}')
print(f'Alias   : {MODEL_ALIAS}')
print(f'Output  : {OUTPUT_FILE}')
print(f'Samples : {NUM_SAMPLES}  |  Max new tokens: {MAX_NEW_TOKENS}')

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    print('WARNING: No GPU. Enable it in Settings → Accelerator.')
else:
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}  VRAM: {total:.1f} GB')

In [ ]:
import glob, os, shutil, json

# Find test_functions.csv in any uploaded dataset
matches = glob.glob('/kaggle/input/**/test_functions.csv', recursive=True)
assert matches, (
    'test_functions.csv not found.\n'
    'Click + Add Input → Datasets → Upload → select test_functions.csv'
)
TEST_CSV_PATH = matches[0]
print(f'Found test data : {TEST_CSV_PATH}')

# Resume: if a partial output from a previous session was uploaded as a dataset, copy it
partial = glob.glob(f'/kaggle/input/**/*{MODEL_ALIAS}*.jsonl', recursive=True)
if partial:
    shutil.copy(partial[0], OUTPUT_FILE)
    lines = sum(1 for _ in open(OUTPUT_FILE))
    print(f'Resuming from partial file: {partial[0]}  ({lines} lines)')
else:
    print('No partial file found — starting fresh.')

In [ ]:
FEWSHOT_DATA = json.loads('{"CWE-119": [{"func_before": " void WebGraphicsContext3DCommandBufferImpl::FlipVertically(\\n     uint8* framebuffer,\\n     unsigned int width,\\n     unsigned int height) {\\n  uint8* scanline = scanline_.get();\\n  if (!scanline)\\n     return;\\n   unsigned int row_bytes = width * 4;\\n   unsigned int count = height / 2;\\n   for (unsigned int i = 0; i < count; i++) {\\n    uint8* row_a = framebuffer + i * row_bytes;\\n    uint8* row_b = framebuffer + (height - i - 1) * row_bytes;\\n    memcpy(scanline, row_b, row_bytes);\\n    memcpy(row_b, row_a, row_bytes);\\n    memcpy(row_a, scanline, row_bytes);\\n  }\\n}\\n", "func_after": " void WebGraphicsContext3DCommandBufferImpl::FlipVertically(\\n     uint8* framebuffer,\\n     unsigned int width,\\n     unsigned int height) {\\n  if (width == 0)\\n     return;\\n  scanline_.resize(width * 4);\\n  uint8* scanline = &scanline_[0];\\n   unsigned int row_bytes = width * 4;\\n   unsigned int count = height / 2;\\n   for (unsigned int i = 0; i < count; i++) {\\n    uint8* row_a = framebuffer + i * row_bytes;\\n    uint8* row_b = framebuffer + (height - i - 1) * row_bytes;\\n    memcpy(scanline, row_b, row_bytes);\\n    memcpy(row_b, row_a, row_bytes);\\n    memcpy(row_a, scanline, row_bytes);\\n  }\\n}\\n"}, {"func_before": "log2vis_utf8 (PyObject * string, int unicode_length,\\n\\t      FriBidiParType base_direction, int clean, int reordernsm)\\n{\\n\\tFriBidiChar *logical = NULL; /* input fribidi unicode buffer */\\n\\tFriBidiChar *visual = NULL;\\t /* output fribidi unicode buffer */\\n\\tchar *visual_utf8 = NULL;    /* output fribidi UTF-8 buffer */\\n\\tFriBidiStrIndex new_len = 0; /* length of the UTF-8 buffer */\\n\\tPyObject *result = NULL;\\t /* failure */\\n\\t/* Allocate fribidi unicode buffers */\\n\\tlogical = PyMem_New (FriBidiChar, unicode_length + 1);\\n\\tif (logical == NULL)\\n\\t{\\n\\t\\tPyErr_SetString (PyExc_MemoryError,\\n\\t\\t\\t\\t \\"failed to allocate unicode buffer\\");\\n\\t\\tgoto cleanup;\\n\\t}\\n\\tvisual = PyMem_New (FriBidiChar, unicode_length + 1);\\n\\tif (visual == NULL)\\n\\t{\\n\\t\\tPyErr_SetString (PyExc_MemoryError,\\n\\t\\t\\t\\t \\"failed to allocate unicode buffer\\");\\n\\t\\tgoto cleanup;\\n\\t}\\n\\t/* Convert to unicode and order visually */\\n\\tfribidi_set_reorder_nsm(reordernsm);\\n\\tfribidi_utf8_to_unicode (PyString_AS_STRING (string),\\n\\t\\t\\t\\t PyString_GET_SIZE (string), logical);\\n\\tif (!fribidi_log2vis (logical, unicode_length, &base_direction, visual,\\n\\t\\t\\t      NULL, NULL, NULL))\\n\\t{\\n\\t\\tPyErr_SetString (PyExc_RuntimeError,\\n\\t\\t\\t\\t \\"fribidi failed to order string\\");\\n\\t\\tgoto cleanup;\\n\\t}\\n\\t/* Cleanup the string if requested */\\n\\tif (clean)\\n\\t\\tfribidi_remove_bidi_marks (visual, unicode_length, NULL, NULL, NULL);\\n\\t/* Allocate fribidi UTF-8 buffer */\\n\\tvisual_utf8 = PyMem_New(char, (unicode_length * 4)+1);\\n\\tif (visual_utf8 == NULL)\\n\\t{\\n\\t\\tPyErr_SetString (PyExc_MemoryError,\\n\\t\\t\\t\\t\\"failed to allocate UTF-8 buffer\\");\\n\\t\\tgoto cleanup;\\n\\t}\\n\\t/* Encode the reordered string  and create result string */\\n\\tnew_len = fribidi_unicode_to_utf8 (visual, unicode_length, visual_utf8);\\n\\tresult = PyString_FromStringAndSize (visual_utf8, new_len);\\n\\tif (result == NULL)\\n\\t\\t/* XXX does it raise any error? */\\n\\t\\tgoto cleanup;\\n      cleanup:\\n\\t/* Delete unicode buffers */\\n\\tPyMem_Del (logical);\\n\\tPyMem_Del (visual);\\n\\tPyMem_Del (visual_utf8);\\n\\treturn result;\\n}\\n", "func_after": "log2vis_utf8 (PyObject * string, int unicode_length,\\n"}, {"func_before": "static void finish_object(struct object *obj,\\n\\t\\t\\t  struct strbuf *path, const char *name,\\n\\t\\t\\t  void *cb_data)\\n {\\n \\tstruct rev_list_info *info = cb_data;\\n \\tif (obj->type == OBJ_BLOB && !has_object_file(&obj->oid))\\n\\t\\tdie(\\"missing blob object \'%s\'\\", oid_to_hex(&obj->oid));\\n\\tif (info->revs->verify_objects && !obj->parsed && obj->type != OBJ_COMMIT)\\n \\t\\tparse_object(obj->oid.hash);\\n }\\n", "func_after": "static void finish_object(struct object *obj,\\nstatic void finish_object(struct object *obj, const char *name, void *cb_data)\\n {\\n \\tstruct rev_list_info *info = cb_data;\\n \\tif (obj->type == OBJ_BLOB && !has_object_file(&obj->oid))\\n\\t\\tdie(\\"missing blob object \'%s\'\\", oid_to_hex(&obj->oid));\\n\\tif (info->revs->verify_objects && !obj->parsed && obj->type != OBJ_COMMIT)\\n \\t\\tparse_object(obj->oid.hash);\\n }\\n"}], "CWE-20": [{"func_before": "long Chapters::Display::Parse(IMkvReader* pReader, long long pos,\\n long long size) {\\n const long long stop = pos + size;\\n\\n while (pos < stop) {\\n long long id, size;\\n\\n long status = ParseElementHeader(pReader, pos, stop, id, size);\\n\\n if (status < 0) // error\\n return status;\\n\\n if (size == 0) // weird\\n continue;\\n\\n if (id == 0x05) { // ChapterString ID\\n      status = UnserializeString(pReader, pos, size, m_string);\\n\\n if (status)\\n return status;\\n } else if (id == 0x037C) { // ChapterLanguage ID\\n      status = UnserializeString(pReader, pos, size, m_language);\\n\\n if (status)\\n return status;\\n } else if (id == 0x037E) { // ChapterCountry ID\\n      status = UnserializeString(pReader, pos, size, m_country);\\n\\n if (status)\\n return status;\\n\\n     }\\n \\n     pos += size;\\n    assert(pos <= stop);\\n   }\\n \\n  assert(pos == stop);\\n   return 0;\\n }\\n", "func_after": "long Chapters::Display::Parse(IMkvReader* pReader, long long pos,\\n long long size) {\\n const long long stop = pos + size;\\n\\n while (pos < stop) {\\n long long id, size;\\n\\n long status = ParseElementHeader(pReader, pos, stop, id, size);\\n\\n if (status < 0) // error\\n return status;\\n\\n if (size == 0) // weird\\n continue;\\n\\n if (id == 0x05) { // ChapterString ID\\n      status = UnserializeString(pReader, pos, size, m_string);\\n\\n if (status)\\n return status;\\n } else if (id == 0x037C) { // ChapterLanguage ID\\n      status = UnserializeString(pReader, pos, size, m_language);\\n\\n if (status)\\n return status;\\n } else if (id == 0x037E) { // ChapterCountry ID\\n      status = UnserializeString(pReader, pos, size, m_country);\\n\\n if (status)\\n return status;\\n\\n     }\\n \\n     pos += size;\\n    if (pos > stop)\\n      return E_FILE_FORMAT_INVALID;\\n   }\\n \\n  if (pos != stop)\\n    return E_FILE_FORMAT_INVALID;\\n  return 0;\\n}\\n\\nTags::Tags(Segment* pSegment, long long payload_start, long long payload_size,\\n           long long element_start, long long element_size)\\n    : m_pSegment(pSegment),\\n      m_start(payload_start),\\n      m_size(payload_size),\\n      m_element_start(element_start),\\n      m_element_size(element_size),\\n      m_tags(NULL),\\n      m_tags_size(0),\\n      m_tags_count(0) {}\\n\\nTags::~Tags() {\\n  while (m_tags_count > 0) {\\n    Tag& t = m_tags[--m_tags_count];\\n    t.Clear();\\n  }\\n  delete[] m_tags;\\n}\\n\\nlong Tags::Parse() {\\n  IMkvReader* const pReader = m_pSegment->m_pReader;\\n\\n  long long pos = m_start;  // payload start\\n  const long long stop = pos + m_size;  // payload stop\\n\\n  while (pos < stop) {\\n    long long id, size;\\n\\n    long status = ParseElementHeader(pReader, pos, stop, id, size);\\n\\n    if (status < 0)\\n      return status;\\n\\n    if (size == 0)  // 0 length tag, read another\\n      continue;\\n\\n    if (id == 0x3373) {  // Tag ID\\n      status = ParseTag(pos, size);\\n\\n      if (status < 0)\\n        return status;\\n    }\\n\\n    pos += size;\\n    if (pos > stop)\\n      return E_FILE_FORMAT_INVALID;\\n  }\\n\\n  if (pos != stop)\\n    return E_FILE_FORMAT_INVALID;\\n\\n  return 0;\\n}\\n\\nint Tags::GetTagCount() const { return m_tags_count; }\\n\\nconst Tags::Tag* Tags::GetTag(int idx) const {\\n  if (idx < 0)\\n    return NULL;\\n\\n  if (idx >= m_tags_count)\\n    return NULL;\\n\\n  return m_tags + idx;\\n}\\n\\nbool Tags::ExpandTagsArray() {\\n  if (m_tags_size > m_tags_count)\\n    return true;  // nothing else to do\\n\\n  const int size = (m_tags_size == 0) ? 1 : 2 * m_tags_size;\\n\\n  Tag* const tags = new (std::nothrow) Tag[size];\\n\\n  if (tags == NULL)\\n    return false;\\n\\n  for (int idx = 0; idx < m_tags_count; ++idx) {\\n    m_tags[idx].ShallowCopy(tags[idx]);\\n  }\\n\\n  delete[] m_tags;\\n  m_tags = tags;\\n\\n  m_tags_size = size;\\n  return true;\\n}\\n\\nlong Tags::ParseTag(long long pos, long long size) {\\n  if (!ExpandTagsArray())\\n    return -1;\\n\\n  Tag& t = m_tags[m_tags_count++];\\n  t.Init();\\n\\n  return t.Parse(m_pSegment->m_pReader, pos, size);\\n}\\n\\nTags::Tag::Tag() {}\\n\\nTags::Tag::~Tag() {}\\n\\nint Tags::Tag::GetSimpleTagCount() const { return m_simple_tags_count; }\\n\\nconst Tags::SimpleTag* Tags::Tag::GetSimpleTag(int index) const {\\n  if (index < 0)\\n    return NULL;\\n\\n  if (index >= m_simple_tags_count)\\n    return NULL;\\n\\n  return m_simple_tags + index;\\n}\\n\\nvoid Tags::Tag::Init() {\\n  m_simple_tags = NULL;\\n  m_simple_tags_size = 0;\\n  m_simple_tags_count = 0;\\n}\\n\\nvoid Tags::Tag::ShallowCopy(Tag& rhs) const {\\n  rhs.m_simple_tags = m_simple_tags;\\n  rhs.m_simple_tags_size = m_simple_tags_size;\\n  rhs.m_simple_tags_count = m_simple_tags_count;\\n}\\n\\nvoid Tags::Tag::Clear() {\\n  while (m_simple_tags_count > 0) {\\n    SimpleTag& d = m_simple_tags[--m_simple_tags_count];\\n    d.Clear();\\n  }\\n\\n  delete[] m_simple_tags;\\n  m_simple_tags = NULL;\\n\\n  m_simple_tags_size = 0;\\n}\\n\\nlong Tags::Tag::Parse(IMkvReader* pReader, long long pos, long long size) {\\n  const long long stop = pos + size;\\n\\n  while (pos < stop) {\\n    long long id, size;\\n\\n    long status = ParseElementHeader(pReader, pos, stop, id, size);\\n\\n    if (status < 0)\\n      return status;\\n\\n    if (size == 0)  // 0 length tag, read another\\n      continue;\\n\\n    if (id == 0x27C8) {  // SimpleTag ID\\n      status = ParseSimpleTag(pReader, pos, size);\\n\\n      if (status < 0)\\n        return status;\\n    }\\n\\n    pos += size;\\n    if (pos > stop)\\n      return E_FILE_FORMAT_INVALID;\\n  }\\n\\n  if (pos != stop)\\n    return E_FILE_FORMAT_INVALID;\\n  return 0;\\n}\\n\\nlong Tags::Tag::ParseSimpleTag(IMkvReader* pReader, long long pos,\\n                               long long size) {\\n  if (!ExpandSimpleTagsArray())\\n    return -1;\\n\\n  SimpleTag& st = m_simple_tags[m_simple_tags_count++];\\n  st.Init();\\n\\n  return st.Parse(pReader, pos, size);\\n}\\n\\nbool Tags::Tag::ExpandSimpleTagsArray() {\\n  if (m_simple_tags_size > m_simple_tags_count)\\n    return true;  // nothing else to do\\n\\n  const int size = (m_simple_tags_size == 0) ? 1 : 2 * m_simple_tags_size;\\n\\n  SimpleTag* const displays = new (std::nothrow) SimpleTag[size];\\n\\n  if (displays == NULL)\\n    return false;\\n\\n  for (int idx = 0; idx < m_simple_tags_count; ++idx) {\\n    m_simple_tags[idx].ShallowCopy(displays[idx]);\\n  }\\n\\n  delete[] m_simple_tags;\\n  m_simple_tags = displays;\\n\\n  m_simple_tags_size = size;\\n  return true;\\n}\\n\\nTags::SimpleTag::SimpleTag() {}\\n\\nTags::SimpleTag::~SimpleTag() {}\\n\\nconst char* Tags::SimpleTag::GetTagName() const { return m_tag_name; }\\n\\nconst char* Tags::SimpleTag::GetTagString() const { return m_tag_string; }\\n\\nvoid Tags::SimpleTag::Init() {\\n  m_tag_name = NULL;\\n  m_tag_string = NULL;\\n}\\n\\nvoid Tags::SimpleTag::ShallowCopy(SimpleTag& rhs) const {\\n  rhs.m_tag_name = m_tag_name;\\n  rhs.m_tag_string = m_tag_string;\\n}\\n\\nvoid Tags::SimpleTag::Clear() {\\n  delete[] m_tag_name;\\n  m_tag_name = NULL;\\n\\n  delete[] m_tag_string;\\n  m_tag_string = NULL;\\n}\\n\\nlong Tags::SimpleTag::Parse(IMkvReader* pReader, long long pos,\\n                            long long size) {\\n  const long long stop = pos + size;\\n\\n  while (pos < stop) {\\n    long long id, size;\\n\\n    long status = ParseElementHeader(pReader, pos, stop, id, size);\\n\\n    if (status < 0)  // error\\n      return status;\\n\\n    if (size == 0)  // weird\\n      continue;\\n\\n    if (id == 0x5A3) {  // TagName ID\\n      status = UnserializeString(pReader, pos, size, m_tag_name);\\n\\n      if (status)\\n        return status;\\n    } else if (id == 0x487) {  // TagString ID\\n      status = UnserializeString(pReader, pos, size, m_tag_string);\\n\\n      if (status)\\n        return status;\\n    }\\n\\n    pos += size;\\n    if (pos > stop)\\n      return E_FILE_FORMAT_INVALID;\\n  }\\n\\n  if (pos != stop)\\n    return E_FILE_FORMAT_INVALID;\\n   return 0;\\n }\\n"}, {"func_before": "void ExtensionTtsController::SetPlatformImpl(\\n    ExtensionTtsPlatformImpl* platform_impl) {\\n  platform_impl_ = platform_impl;\\n}\\n", "func_after": "void ExtensionTtsController::SetPlatformImpl(\\n"}, {"func_before": "static ssize_t yurex_read(struct file *file, char __user *buffer, size_t count,\\n \\t\\t\\t  loff_t *ppos)\\n {\\n \\tstruct usb_yurex *dev;\\n\\tint retval = 0;\\n\\tint bytes_read = 0;\\n \\tchar in_buffer[20];\\n \\tunsigned long flags;\\n \\n \\tdev = file->private_data;\\n \\n \\tmutex_lock(&dev->io_mutex);\\n \\tif (!dev->interface) {\\t\\t/* already disconnected */\\n\\t\\tretval = -ENODEV;\\n\\t\\tgoto exit;\\n \\t}\\n \\n \\tspin_lock_irqsave(&dev->lock, flags);\\n\\tbytes_read = snprintf(in_buffer, 20, \\"%lld\\\\n\\", dev->bbu);\\n \\tspin_unlock_irqrestore(&dev->lock, flags);\\n\\tif (*ppos < bytes_read) {\\n\\t\\tif (copy_to_user(buffer, in_buffer + *ppos, bytes_read - *ppos))\\n\\t\\t\\tretval = -EFAULT;\\n\\t\\telse {\\n\\t\\t\\tretval = bytes_read - *ppos;\\n\\t\\t\\t*ppos += bytes_read;\\n\\t\\t}\\n\\t}\\nexit:\\n \\tmutex_unlock(&dev->io_mutex);\\n\\treturn retval;\\n }\\n", "func_after": "static ssize_t yurex_read(struct file *file, char __user *buffer, size_t count,\\n \\t\\t\\t  loff_t *ppos)\\n {\\n \\tstruct usb_yurex *dev;\\n\\tint len = 0;\\n \\tchar in_buffer[20];\\n \\tunsigned long flags;\\n \\n \\tdev = file->private_data;\\n \\n \\tmutex_lock(&dev->io_mutex);\\n \\tif (!dev->interface) {\\t\\t/* already disconnected */\\n\\t\\tmutex_unlock(&dev->io_mutex);\\n\\t\\treturn -ENODEV;\\n \\t}\\n \\n \\tspin_lock_irqsave(&dev->lock, flags);\\n\\tlen = snprintf(in_buffer, 20, \\"%lld\\\\n\\", dev->bbu);\\n \\tspin_unlock_irqrestore(&dev->lock, flags);\\n \\tmutex_unlock(&dev->io_mutex);\\n\\n\\treturn simple_read_from_buffer(buffer, count, ppos, in_buffer, len);\\n }\\n"}], "CWE-399": [{"func_before": "void XMLHttpRequest::didTimeout()\\n {\\n     RefPtr<XMLHttpRequest> protect(this);\\n     internalAbort();\\n \\n    clearResponse();\\n    clearRequest();\\n    m_error = true;\\n     m_exceptionCode = TimeoutError;\\n \\n     if (!m_async) {\\n         m_state = DONE;\\n        m_exceptionCode = TimeoutError;\\n         return;\\n     }\\n     changeState(DONE);\\n \\n    if (!m_uploadComplete) {\\n        m_uploadComplete = true;\\n        if (m_upload && m_uploadEventsAllowed)\\n            m_upload->dispatchEventAndLoadEnd(XMLHttpRequestProgressEvent::create(eventNames().timeoutEvent));\\n    }\\n    m_progressEventThrottle.dispatchEventAndLoadEnd(XMLHttpRequestProgressEvent::create(eventNames().timeoutEvent));\\n }\\n", "func_after": "void XMLHttpRequest::didTimeout()\\nvoid XMLHttpRequest::handleDidTimeout()\\n {\\n     RefPtr<XMLHttpRequest> protect(this);\\n     internalAbort();\\n \\n     m_exceptionCode = TimeoutError;\\n \\n    handleDidFailGeneric();\\n\\n     if (!m_async) {\\n         m_state = DONE;\\n         return;\\n     }\\n     changeState(DONE);\\n \\n    dispatchEventAndLoadEnd(eventNames().timeoutEvent);\\n }\\n"}, {"func_before": "static int asf_build_simple_index(AVFormatContext *s, int stream_index)\\n{\\n    ff_asf_guid g;\\n    ASFContext *asf     = s->priv_data;\\n    int64_t current_pos = avio_tell(s->pb);\\n    int64_t ret;\\n\\n    if((ret = avio_seek(s->pb, asf->data_object_offset + asf->data_object_size, SEEK_SET)) < 0) {\\n        return ret;\\n    }\\n\\n    if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n        goto end;\\n\\n    /* the data object can be followed by other top-level objects,\\n     * skip them until the simple index object is reached */\\n    while (ff_guidcmp(&g, &ff_asf_simple_index_header)) {\\n        int64_t gsize = avio_rl64(s->pb);\\n        if (gsize < 24 || avio_feof(s->pb)) {\\n            goto end;\\n        }\\n        avio_skip(s->pb, gsize - 24);\\n        if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n            goto end;\\n    }\\n\\n    {\\n        int64_t itime, last_pos = -1;\\n        int pct, ict;\\n        int i;\\n        int64_t av_unused gsize = avio_rl64(s->pb);\\n        if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n            goto end;\\n        itime = avio_rl64(s->pb);\\n        pct   = avio_rl32(s->pb);\\n        ict   = avio_rl32(s->pb);\\n        av_log(s, AV_LOG_DEBUG,\\n               \\"itime:0x%\\"PRIx64\\", pct:%d, ict:%d\\\\n\\", itime, pct, ict);\\n\\n        for (i = 0; i < ict; i++) {\\n            int pktnum        = avio_rl32(s->pb);\\n            int pktct         = avio_rl16(s->pb);\\n             int64_t pos       = s->internal->data_offset + s->packet_size * (int64_t)pktnum;\\n             int64_t index_pts = FFMAX(av_rescale(itime, i, 10000) - asf->hdr.preroll, 0);\\n \\n             if (pos != last_pos) {\\n                 av_log(s, AV_LOG_DEBUG, \\"pktnum:%d, pktct:%d  pts: %\\"PRId64\\"\\\\n\\",\\n                        pktnum, pktct, index_pts);\\n                av_add_index_entry(s->streams[stream_index], pos, index_pts,\\n                                   s->packet_size, 0, AVINDEX_KEYFRAME);\\n                last_pos = pos;\\n            }\\n        }\\n        asf->index_read = ict > 1;\\n    }\\nend:\\n    avio_seek(s->pb, current_pos, SEEK_SET);\\n    return ret;\\n}\\n", "func_after": "static int asf_build_simple_index(AVFormatContext *s, int stream_index)\\n{\\n    ff_asf_guid g;\\n    ASFContext *asf     = s->priv_data;\\n    int64_t current_pos = avio_tell(s->pb);\\n    int64_t ret;\\n\\n    if((ret = avio_seek(s->pb, asf->data_object_offset + asf->data_object_size, SEEK_SET)) < 0) {\\n        return ret;\\n    }\\n\\n    if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n        goto end;\\n\\n    /* the data object can be followed by other top-level objects,\\n     * skip them until the simple index object is reached */\\n    while (ff_guidcmp(&g, &ff_asf_simple_index_header)) {\\n        int64_t gsize = avio_rl64(s->pb);\\n        if (gsize < 24 || avio_feof(s->pb)) {\\n            goto end;\\n        }\\n        avio_skip(s->pb, gsize - 24);\\n        if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n            goto end;\\n    }\\n\\n    {\\n        int64_t itime, last_pos = -1;\\n        int pct, ict;\\n        int i;\\n        int64_t av_unused gsize = avio_rl64(s->pb);\\n        if ((ret = ff_get_guid(s->pb, &g)) < 0)\\n            goto end;\\n        itime = avio_rl64(s->pb);\\n        pct   = avio_rl32(s->pb);\\n        ict   = avio_rl32(s->pb);\\n        av_log(s, AV_LOG_DEBUG,\\n               \\"itime:0x%\\"PRIx64\\", pct:%d, ict:%d\\\\n\\", itime, pct, ict);\\n\\n        for (i = 0; i < ict; i++) {\\n            int pktnum        = avio_rl32(s->pb);\\n            int pktct         = avio_rl16(s->pb);\\n             int64_t pos       = s->internal->data_offset + s->packet_size * (int64_t)pktnum;\\n             int64_t index_pts = FFMAX(av_rescale(itime, i, 10000) - asf->hdr.preroll, 0);\\n \\n            if (avio_feof(s->pb)) {\\n                ret = AVERROR_INVALIDDATA;\\n                goto end;\\n            }\\n\\n             if (pos != last_pos) {\\n                 av_log(s, AV_LOG_DEBUG, \\"pktnum:%d, pktct:%d  pts: %\\"PRId64\\"\\\\n\\",\\n                        pktnum, pktct, index_pts);\\n                av_add_index_entry(s->streams[stream_index], pos, index_pts,\\n                                   s->packet_size, 0, AVINDEX_KEYFRAME);\\n                last_pos = pos;\\n            }\\n        }\\n        asf->index_read = ict > 1;\\n    }\\nend:\\n    avio_seek(s->pb, current_pos, SEEK_SET);\\n    return ret;\\n}\\n"}, {"func_before": "GDataFileError GDataWapiFeedProcessor::FeedToFileResourceMap(\\n    const std::vector<DocumentFeed*>& feed_list,\\n    FileResourceIdMap* file_map,\\n    int64* feed_changestamp,\\n    FeedToFileResourceMapUmaStats* uma_stats) {\\n  DCHECK(BrowserThread::CurrentlyOn(BrowserThread::UI));\\n  DCHECK(uma_stats);\\n\\n  GDataFileError error = GDATA_FILE_OK;\\n  uma_stats->num_regular_files = 0;\\n  uma_stats->num_hosted_documents = 0;\\n  uma_stats->num_files_with_entry_kind.clear();\\n  for (size_t i = 0; i < feed_list.size(); ++i) {\\n    const DocumentFeed* feed = feed_list[i];\\n\\n    if (i == 0) {\\n      const Link* root_feed_upload_link =\\n          feed->GetLinkByType(Link::RESUMABLE_CREATE_MEDIA);\\n      if (root_feed_upload_link)\\n        directory_service_->root()->set_upload_url(\\n            root_feed_upload_link->href());\\n      *feed_changestamp = feed->largest_changestamp();\\n      DCHECK_GE(*feed_changestamp, 0);\\n    }\\n\\n    for (ScopedVector<DocumentEntry>::const_iterator iter =\\n              feed->entries().begin();\\n          iter != feed->entries().end(); ++iter) {\\n       DocumentEntry* doc = *iter;\\n      GDataEntry* entry = GDataEntry::FromDocumentEntry(\\n          NULL, doc, directory_service_);\\n       if (!entry)\\n         continue;\\n      GDataFile* as_file = entry->AsGDataFile();\\n      if (as_file) {\\n        if (as_file->is_hosted_document())\\n          ++uma_stats->num_hosted_documents;\\n        else\\n          ++uma_stats->num_regular_files;\\n        ++uma_stats->num_files_with_entry_kind[as_file->kind()];\\n      }\\n\\n      FileResourceIdMap::iterator map_entry =\\n          file_map->find(entry->resource_id());\\n\\n      if (map_entry != file_map->end()) {\\n        LOG(WARNING) << \\"Found duplicate file \\"\\n                     << map_entry->second->base_name();\\n\\n        delete map_entry->second;\\n        file_map->erase(map_entry);\\n      }\\n      file_map->insert(\\n          std::pair<std::string, GDataEntry*>(entry->resource_id(), entry));\\n    }\\n  }\\n\\n  if (error != GDATA_FILE_OK) {\\n    STLDeleteValues(file_map);\\n  }\\n\\n  return error;\\n}\\n", "func_after": "GDataFileError GDataWapiFeedProcessor::FeedToFileResourceMap(\\n    const std::vector<DocumentFeed*>& feed_list,\\n    FileResourceIdMap* file_map,\\n    int64* feed_changestamp,\\n    FeedToFileResourceMapUmaStats* uma_stats) {\\n  DCHECK(BrowserThread::CurrentlyOn(BrowserThread::UI));\\n  DCHECK(uma_stats);\\n\\n  GDataFileError error = GDATA_FILE_OK;\\n  uma_stats->num_regular_files = 0;\\n  uma_stats->num_hosted_documents = 0;\\n  uma_stats->num_files_with_entry_kind.clear();\\n  for (size_t i = 0; i < feed_list.size(); ++i) {\\n    const DocumentFeed* feed = feed_list[i];\\n\\n    if (i == 0) {\\n      const Link* root_feed_upload_link =\\n          feed->GetLinkByType(Link::RESUMABLE_CREATE_MEDIA);\\n      if (root_feed_upload_link)\\n        directory_service_->root()->set_upload_url(\\n            root_feed_upload_link->href());\\n      *feed_changestamp = feed->largest_changestamp();\\n      DCHECK_GE(*feed_changestamp, 0);\\n    }\\n\\n    for (ScopedVector<DocumentEntry>::const_iterator iter =\\n              feed->entries().begin();\\n          iter != feed->entries().end(); ++iter) {\\n       DocumentEntry* doc = *iter;\\n      GDataEntry* entry = directory_service_->FromDocumentEntry(doc);\\n       if (!entry)\\n         continue;\\n      GDataFile* as_file = entry->AsGDataFile();\\n      if (as_file) {\\n        if (as_file->is_hosted_document())\\n          ++uma_stats->num_hosted_documents;\\n        else\\n          ++uma_stats->num_regular_files;\\n        ++uma_stats->num_files_with_entry_kind[as_file->kind()];\\n      }\\n\\n      FileResourceIdMap::iterator map_entry =\\n          file_map->find(entry->resource_id());\\n\\n      if (map_entry != file_map->end()) {\\n        LOG(WARNING) << \\"Found duplicate file \\"\\n                     << map_entry->second->base_name();\\n\\n        delete map_entry->second;\\n        file_map->erase(map_entry);\\n      }\\n      file_map->insert(\\n          std::pair<std::string, GDataEntry*>(entry->resource_id(), entry));\\n    }\\n  }\\n\\n  if (error != GDATA_FILE_OK) {\\n    STLDeleteValues(file_map);\\n  }\\n\\n  return error;\\n}\\n"}], "CWE-264": [{"func_before": "static int multipath_ioctl(struct dm_target *ti, unsigned int cmd,\\n\\t\\t\\t   unsigned long arg)\\n{\\n\\tstruct multipath *m = (struct multipath *) ti->private;\\n\\tstruct block_device *bdev = NULL;\\n\\tfmode_t mode = 0;\\n\\tunsigned long flags;\\n\\tint r = 0;\\n\\n\\tspin_lock_irqsave(&m->lock, flags);\\n\\n\\tif (!m->current_pgpath)\\n\\t\\t__choose_pgpath(m, 0);\\n\\n\\tif (m->current_pgpath) {\\n\\t\\tbdev = m->current_pgpath->path.dev->bdev;\\n\\t\\tmode = m->current_pgpath->path.dev->mode;\\n\\t}\\n\\n\\tif (m->queue_io)\\n\\t\\tr = -EAGAIN;\\n\\telse if (!bdev)\\n\\t\\tr = -EIO;\\n \\n \\tspin_unlock_irqrestore(&m->lock, flags);\\n \\n \\treturn r ? : __blkdev_driver_ioctl(bdev, mode, cmd, arg);\\n }\\n", "func_after": "static int multipath_ioctl(struct dm_target *ti, unsigned int cmd,\\n\\t\\t\\t   unsigned long arg)\\n{\\n\\tstruct multipath *m = (struct multipath *) ti->private;\\n\\tstruct block_device *bdev = NULL;\\n\\tfmode_t mode = 0;\\n\\tunsigned long flags;\\n\\tint r = 0;\\n\\n\\tspin_lock_irqsave(&m->lock, flags);\\n\\n\\tif (!m->current_pgpath)\\n\\t\\t__choose_pgpath(m, 0);\\n\\n\\tif (m->current_pgpath) {\\n\\t\\tbdev = m->current_pgpath->path.dev->bdev;\\n\\t\\tmode = m->current_pgpath->path.dev->mode;\\n\\t}\\n\\n\\tif (m->queue_io)\\n\\t\\tr = -EAGAIN;\\n\\telse if (!bdev)\\n\\t\\tr = -EIO;\\n \\n \\tspin_unlock_irqrestore(&m->lock, flags);\\n \\n\\t/*\\n\\t * Only pass ioctls through if the device sizes match exactly.\\n\\t */\\n\\tif (!r && ti->len != i_size_read(bdev->bd_inode) >> SECTOR_SHIFT)\\n\\t\\tr = scsi_verify_blk_ioctl(NULL, cmd);\\n\\n \\treturn r ? : __blkdev_driver_ioctl(bdev, mode, cmd, arg);\\n }\\n"}, {"func_before": " bool InputWindowInfo::isTrustedOverlay() const {\\n     return layoutParamsType == TYPE_INPUT_METHOD\\n             || layoutParamsType == TYPE_INPUT_METHOD_DIALOG\\n             || layoutParamsType == TYPE_MAGNIFICATION_OVERLAY\\n             || layoutParamsType == TYPE_SECURE_SYSTEM_OVERLAY;\\n }\\n", "func_after": " bool InputWindowInfo::isTrustedOverlay() const {\\n     return layoutParamsType == TYPE_INPUT_METHOD\\n             || layoutParamsType == TYPE_INPUT_METHOD_DIALOG\\n             || layoutParamsType == TYPE_MAGNIFICATION_OVERLAY\\n            || layoutParamsType == TYPE_STATUS_BAR\\n            || layoutParamsType == TYPE_NAVIGATION_BAR\\n             || layoutParamsType == TYPE_SECURE_SYSTEM_OVERLAY;\\n }\\n"}, {"func_before": "_fep_open_control_socket (Fep *fep)\\n{\\n  struct sockaddr_un sun;\\n  char *path;\\n  int fd;\\n  ssize_t sun_len;\\n\\n  fd = socket (AF_UNIX, SOCK_STREAM, 0);\\n  if (fd < 0)\\n    {\\n      perror (\\"socket\\");\\n      return -1;\\n    }\\n\\n  path = create_socket_name (\\"fep-XXXXXX/control\\");\\n  if (strlen (path) + 1 >= sizeof(sun.sun_path))\\n    {\\n      fep_log (FEP_LOG_LEVEL_WARNING,\\n\\t       \\"unix domain socket path too long: %d + 1 >= %d\\",\\n\\t       strlen (path),\\n\\t       sizeof (sun.sun_path));\\n      free (path);\\n      return -1;\\n    }\\n\\n   memset (&sun, 0, sizeof(sun));\\n   sun.sun_family = AF_UNIX;\\n \\n#ifdef __linux__\\n  sun.sun_path[0] = \'\\\\0\';\\n  memcpy (sun.sun_path + 1, path, strlen (path));\\n  sun_len = offsetof (struct sockaddr_un, sun_path) + strlen (path) + 1;\\n  remove_control_socket (path);\\n#else\\n   memcpy (sun.sun_path, path, strlen (path));\\n   sun_len = sizeof (struct sockaddr_un);\\n#endif\\n \\n   if (bind (fd, (const struct sockaddr *) &sun, sun_len) < 0)\\n     {\\n      perror (\\"bind\\");\\n      free (path);\\n      close (fd);\\n      return -1;\\n    }\\n\\n  if (listen (fd, 5) < 0)\\n    {\\n      perror (\\"listen\\");\\n      free (path);\\n      close (fd);\\n      return -1;\\n    }\\n\\n  fep->server = fd;\\n  fep->control_socket_path = path;\\n  return 0;\\n}\\n", "func_after": "_fep_open_control_socket (Fep *fep)\\n{\\n  struct sockaddr_un sun;\\n  char *path;\\n  int fd;\\n  ssize_t sun_len;\\n\\n  fd = socket (AF_UNIX, SOCK_STREAM, 0);\\n  if (fd < 0)\\n    {\\n      perror (\\"socket\\");\\n      return -1;\\n    }\\n\\n  path = create_socket_name (\\"fep-XXXXXX/control\\");\\n  if (strlen (path) + 1 >= sizeof(sun.sun_path))\\n    {\\n      fep_log (FEP_LOG_LEVEL_WARNING,\\n\\t       \\"unix domain socket path too long: %d + 1 >= %d\\",\\n\\t       strlen (path),\\n\\t       sizeof (sun.sun_path));\\n      free (path);\\n      return -1;\\n    }\\n\\n   memset (&sun, 0, sizeof(sun));\\n   sun.sun_family = AF_UNIX;\\n \\n   memcpy (sun.sun_path, path, strlen (path));\\n   sun_len = sizeof (struct sockaddr_un);\\n \\n   if (bind (fd, (const struct sockaddr *) &sun, sun_len) < 0)\\n     {\\n      perror (\\"bind\\");\\n      free (path);\\n      close (fd);\\n      return -1;\\n    }\\n\\n  if (listen (fd, 5) < 0)\\n    {\\n      perror (\\"listen\\");\\n      free (path);\\n      close (fd);\\n      return -1;\\n    }\\n\\n  fep->server = fd;\\n  fep->control_socket_path = path;\\n  return 0;\\n}\\n"}], "CWE-200": [{"func_before": "static int l2tp_ip6_getname(struct socket *sock, struct sockaddr *uaddr,\\n\\t\\t\\t    int *uaddr_len, int peer)\\n{\\n\\tstruct sockaddr_l2tpip6 *lsa = (struct sockaddr_l2tpip6 *)uaddr;\\n\\tstruct sock *sk = sock->sk;\\n\\tstruct ipv6_pinfo *np = inet6_sk(sk);\\n\\tstruct l2tp_ip6_sock *lsk = l2tp_ip6_sk(sk);\\n\\n \\tlsa->l2tp_family = AF_INET6;\\n \\tlsa->l2tp_flowinfo = 0;\\n \\tlsa->l2tp_scope_id = 0;\\n \\tif (peer) {\\n \\t\\tif (!lsk->peer_conn_id)\\n \\t\\t\\treturn -ENOTCONN;\\n\\t\\tlsa->l2tp_conn_id = lsk->peer_conn_id;\\n\\t\\tlsa->l2tp_addr = np->daddr;\\n\\t\\tif (np->sndflow)\\n\\t\\t\\tlsa->l2tp_flowinfo = np->flow_label;\\n\\t} else {\\n\\t\\tif (ipv6_addr_any(&np->rcv_saddr))\\n\\t\\t\\tlsa->l2tp_addr = np->saddr;\\n\\t\\telse\\n\\t\\t\\tlsa->l2tp_addr = np->rcv_saddr;\\n\\n\\t\\tlsa->l2tp_conn_id = lsk->conn_id;\\n\\t}\\n\\tif (ipv6_addr_type(&lsa->l2tp_addr) & IPV6_ADDR_LINKLOCAL)\\n\\t\\tlsa->l2tp_scope_id = sk->sk_bound_dev_if;\\n\\t*uaddr_len = sizeof(*lsa);\\n\\treturn 0;\\n}\\n", "func_after": "static int l2tp_ip6_getname(struct socket *sock, struct sockaddr *uaddr,\\n\\t\\t\\t    int *uaddr_len, int peer)\\n{\\n\\tstruct sockaddr_l2tpip6 *lsa = (struct sockaddr_l2tpip6 *)uaddr;\\n\\tstruct sock *sk = sock->sk;\\n\\tstruct ipv6_pinfo *np = inet6_sk(sk);\\n\\tstruct l2tp_ip6_sock *lsk = l2tp_ip6_sk(sk);\\n\\n \\tlsa->l2tp_family = AF_INET6;\\n \\tlsa->l2tp_flowinfo = 0;\\n \\tlsa->l2tp_scope_id = 0;\\n\\tlsa->l2tp_unused = 0;\\n \\tif (peer) {\\n \\t\\tif (!lsk->peer_conn_id)\\n \\t\\t\\treturn -ENOTCONN;\\n\\t\\tlsa->l2tp_conn_id = lsk->peer_conn_id;\\n\\t\\tlsa->l2tp_addr = np->daddr;\\n\\t\\tif (np->sndflow)\\n\\t\\t\\tlsa->l2tp_flowinfo = np->flow_label;\\n\\t} else {\\n\\t\\tif (ipv6_addr_any(&np->rcv_saddr))\\n\\t\\t\\tlsa->l2tp_addr = np->saddr;\\n\\t\\telse\\n\\t\\t\\tlsa->l2tp_addr = np->rcv_saddr;\\n\\n\\t\\tlsa->l2tp_conn_id = lsk->conn_id;\\n\\t}\\n\\tif (ipv6_addr_type(&lsa->l2tp_addr) & IPV6_ADDR_LINKLOCAL)\\n\\t\\tlsa->l2tp_scope_id = sk->sk_bound_dev_if;\\n\\t*uaddr_len = sizeof(*lsa);\\n\\treturn 0;\\n}\\n"}, {"func_before": "bool ExtensionApiTest::InitializeEmbeddedTestServer() {\\n  if (!embedded_test_server()->InitializeAndListen())\\n    return false;\\n\\n  test_config_->SetInteger(kEmbeddedTestServerPort,\\n                           embedded_test_server()->port());\\n \\n   return true;\\n }\\n", "func_after": "bool ExtensionApiTest::InitializeEmbeddedTestServer() {\\n  if (!embedded_test_server()->InitializeAndListen())\\n    return false;\\n\\n  if (test_config_) {\\n    test_config_->SetInteger(kEmbeddedTestServerPort,\\n                             embedded_test_server()->port());\\n  }\\n  // else SetUpOnMainThread has not been called yet. Possibly because the\\n  // caller needs a valid port in an overridden SetUpCommandLine method.\\n \\n   return true;\\n }\\n"}, {"func_before": "void DevToolsDataSource::StartDataRequest(\\n    const std::string& path,\\n    const content::ResourceRequestInfo::WebContentsGetter& wc_getter,\\n    const content::URLDataSource::GotDataCallback& callback) {\\n  std::string bundled_path_prefix(chrome::kChromeUIDevToolsBundledPath);\\n  bundled_path_prefix += \\"/\\";\\n  if (base::StartsWith(path, bundled_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    StartBundledDataRequest(path.substr(bundled_path_prefix.length()),\\n                            callback);\\n    return;\\n  }\\n\\n  std::string empty_path_prefix(chrome::kChromeUIDevToolsBlankPath);\\n  if (base::StartsWith(path, empty_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    callback.Run(new base::RefCountedStaticMemory());\\n    return;\\n  }\\n\\n  std::string remote_path_prefix(chrome::kChromeUIDevToolsRemotePath);\\n  remote_path_prefix += \\"/\\";\\n  if (base::StartsWith(path, remote_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    GURL url(kRemoteFrontendBase + path.substr(remote_path_prefix.length()));\\n\\n    CHECK_EQ(url.host(), kRemoteFrontendDomain);\\n    if (url.is_valid() && DevToolsUIBindings::IsValidRemoteFrontendURL(url)) {\\n      StartRemoteDataRequest(url, callback);\\n    } else {\\n      DLOG(ERROR) << \\"Refusing to load invalid remote front-end URL\\";\\n      callback.Run(new base::RefCountedStaticMemory(kHttpNotFound,\\n                                                    strlen(kHttpNotFound)));\\n    }\\n    return;\\n  }\\n\\n  std::string custom_frontend_url =\\n      base::CommandLine::ForCurrentProcess()->GetSwitchValueASCII(\\n          switches::kCustomDevtoolsFrontend);\\n\\n  if (custom_frontend_url.empty()) {\\n    callback.Run(NULL);\\n    return;\\n  }\\n\\n  std::string custom_path_prefix(chrome::kChromeUIDevToolsCustomPath);\\n  custom_path_prefix += \\"/\\";\\n\\n  if (base::StartsWith(path, custom_path_prefix,\\n                        base::CompareCase::INSENSITIVE_ASCII)) {\\n     GURL url = GURL(custom_frontend_url +\\n                     path.substr(custom_path_prefix.length()));\\n     StartCustomDataRequest(url, callback);\\n     return;\\n   }\\n\\n  callback.Run(NULL);\\n}\\n", "func_after": "void DevToolsDataSource::StartDataRequest(\\n    const std::string& path,\\n    const content::ResourceRequestInfo::WebContentsGetter& wc_getter,\\n    const content::URLDataSource::GotDataCallback& callback) {\\n  std::string bundled_path_prefix(chrome::kChromeUIDevToolsBundledPath);\\n  bundled_path_prefix += \\"/\\";\\n  if (base::StartsWith(path, bundled_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    StartBundledDataRequest(path.substr(bundled_path_prefix.length()),\\n                            callback);\\n    return;\\n  }\\n\\n  std::string empty_path_prefix(chrome::kChromeUIDevToolsBlankPath);\\n  if (base::StartsWith(path, empty_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    callback.Run(new base::RefCountedStaticMemory());\\n    return;\\n  }\\n\\n  std::string remote_path_prefix(chrome::kChromeUIDevToolsRemotePath);\\n  remote_path_prefix += \\"/\\";\\n  if (base::StartsWith(path, remote_path_prefix,\\n                       base::CompareCase::INSENSITIVE_ASCII)) {\\n    GURL url(kRemoteFrontendBase + path.substr(remote_path_prefix.length()));\\n\\n    CHECK_EQ(url.host(), kRemoteFrontendDomain);\\n    if (url.is_valid() && DevToolsUIBindings::IsValidRemoteFrontendURL(url)) {\\n      StartRemoteDataRequest(url, callback);\\n    } else {\\n      DLOG(ERROR) << \\"Refusing to load invalid remote front-end URL\\";\\n      callback.Run(new base::RefCountedStaticMemory(kHttpNotFound,\\n                                                    strlen(kHttpNotFound)));\\n    }\\n    return;\\n  }\\n\\n  std::string custom_frontend_url =\\n      base::CommandLine::ForCurrentProcess()->GetSwitchValueASCII(\\n          switches::kCustomDevtoolsFrontend);\\n\\n  if (custom_frontend_url.empty()) {\\n    callback.Run(NULL);\\n    return;\\n  }\\n\\n  std::string custom_path_prefix(chrome::kChromeUIDevToolsCustomPath);\\n  custom_path_prefix += \\"/\\";\\n\\n  if (base::StartsWith(path, custom_path_prefix,\\n                        base::CompareCase::INSENSITIVE_ASCII)) {\\n     GURL url = GURL(custom_frontend_url +\\n                     path.substr(custom_path_prefix.length()));\\n    DCHECK(url.is_valid());\\n     StartCustomDataRequest(url, callback);\\n     return;\\n   }\\n\\n  callback.Run(NULL);\\n}\\n"}], "CWE-125": [{"func_before": "static int usbhid_parse(struct hid_device *hid)\\n{\\n\\tstruct usb_interface *intf = to_usb_interface(hid->dev.parent);\\n\\tstruct usb_host_interface *interface = intf->cur_altsetting;\\n\\tstruct usb_device *dev = interface_to_usbdev (intf);\\n\\tstruct hid_descriptor *hdesc;\\n\\tu32 quirks = 0;\\n \\tunsigned int rsize = 0;\\n \\tchar *rdesc;\\n \\tint ret, n;\\n \\n \\tquirks = usbhid_lookup_quirk(le16_to_cpu(dev->descriptor.idVendor),\\n \\t\\t\\tle16_to_cpu(dev->descriptor.idProduct));\\n\\n\\tif (quirks & HID_QUIRK_IGNORE)\\n\\t\\treturn -ENODEV;\\n\\n\\t/* Many keyboards and mice don\'t like to be polled for reports,\\n\\t * so we will always set the HID_QUIRK_NOGET flag for them. */\\n\\tif (interface->desc.bInterfaceSubClass == USB_INTERFACE_SUBCLASS_BOOT) {\\n\\t\\tif (interface->desc.bInterfaceProtocol == USB_INTERFACE_PROTOCOL_KEYBOARD ||\\n\\t\\t\\tinterface->desc.bInterfaceProtocol == USB_INTERFACE_PROTOCOL_MOUSE)\\n\\t\\t\\t\\tquirks |= HID_QUIRK_NOGET;\\n\\t}\\n\\n\\tif (usb_get_extra_descriptor(interface, HID_DT_HID, &hdesc) &&\\n\\t    (!interface->desc.bNumEndpoints ||\\n\\t     usb_get_extra_descriptor(&interface->endpoint[0], HID_DT_HID, &hdesc))) {\\n\\t\\tdbg_hid(\\"class descriptor not present\\\\n\\");\\n \\t\\treturn -ENODEV;\\n \\t}\\n \\n \\thid->version = le16_to_cpu(hdesc->bcdHID);\\n \\thid->country = hdesc->bCountryCode;\\n \\n\\tfor (n = 0; n < hdesc->bNumDescriptors; n++)\\n \\t\\tif (hdesc->desc[n].bDescriptorType == HID_DT_REPORT)\\n \\t\\t\\trsize = le16_to_cpu(hdesc->desc[n].wDescriptorLength);\\n \\n\\tif (!rsize || rsize > HID_MAX_DESCRIPTOR_SIZE) {\\n\\t\\tdbg_hid(\\"weird size of report descriptor (%u)\\\\n\\", rsize);\\n\\t\\treturn -EINVAL;\\n\\t}\\n\\n\\trdesc = kmalloc(rsize, GFP_KERNEL);\\n\\tif (!rdesc)\\n\\t\\treturn -ENOMEM;\\n\\n\\thid_set_idle(dev, interface->desc.bInterfaceNumber, 0, 0);\\n\\n\\tret = hid_get_class_descriptor(dev, interface->desc.bInterfaceNumber,\\n\\t\\t\\tHID_DT_REPORT, rdesc, rsize);\\n\\tif (ret < 0) {\\n\\t\\tdbg_hid(\\"reading report descriptor failed\\\\n\\");\\n\\t\\tkfree(rdesc);\\n\\t\\tgoto err;\\n\\t}\\n\\n\\tret = hid_parse_report(hid, rdesc, rsize);\\n\\tkfree(rdesc);\\n\\tif (ret) {\\n\\t\\tdbg_hid(\\"parsing report descriptor failed\\\\n\\");\\n\\t\\tgoto err;\\n\\t}\\n\\n\\thid->quirks |= quirks;\\n\\n\\treturn 0;\\nerr:\\n\\treturn ret;\\n}\\n", "func_after": "static int usbhid_parse(struct hid_device *hid)\\n{\\n\\tstruct usb_interface *intf = to_usb_interface(hid->dev.parent);\\n\\tstruct usb_host_interface *interface = intf->cur_altsetting;\\n\\tstruct usb_device *dev = interface_to_usbdev (intf);\\n\\tstruct hid_descriptor *hdesc;\\n\\tu32 quirks = 0;\\n \\tunsigned int rsize = 0;\\n \\tchar *rdesc;\\n \\tint ret, n;\\n\\tint num_descriptors;\\n\\tsize_t offset = offsetof(struct hid_descriptor, desc);\\n \\n \\tquirks = usbhid_lookup_quirk(le16_to_cpu(dev->descriptor.idVendor),\\n \\t\\t\\tle16_to_cpu(dev->descriptor.idProduct));\\n\\n\\tif (quirks & HID_QUIRK_IGNORE)\\n\\t\\treturn -ENODEV;\\n\\n\\t/* Many keyboards and mice don\'t like to be polled for reports,\\n\\t * so we will always set the HID_QUIRK_NOGET flag for them. */\\n\\tif (interface->desc.bInterfaceSubClass == USB_INTERFACE_SUBCLASS_BOOT) {\\n\\t\\tif (interface->desc.bInterfaceProtocol == USB_INTERFACE_PROTOCOL_KEYBOARD ||\\n\\t\\t\\tinterface->desc.bInterfaceProtocol == USB_INTERFACE_PROTOCOL_MOUSE)\\n\\t\\t\\t\\tquirks |= HID_QUIRK_NOGET;\\n\\t}\\n\\n\\tif (usb_get_extra_descriptor(interface, HID_DT_HID, &hdesc) &&\\n\\t    (!interface->desc.bNumEndpoints ||\\n\\t     usb_get_extra_descriptor(&interface->endpoint[0], HID_DT_HID, &hdesc))) {\\n\\t\\tdbg_hid(\\"class descriptor not present\\\\n\\");\\n \\t\\treturn -ENODEV;\\n \\t}\\n \\n\\tif (hdesc->bLength < sizeof(struct hid_descriptor)) {\\n\\t\\tdbg_hid(\\"hid descriptor is too short\\\\n\\");\\n\\t\\treturn -EINVAL;\\n\\t}\\n\\n \\thid->version = le16_to_cpu(hdesc->bcdHID);\\n \\thid->country = hdesc->bCountryCode;\\n \\n\\tnum_descriptors = min_t(int, hdesc->bNumDescriptors,\\n\\t       (hdesc->bLength - offset) / sizeof(struct hid_class_descriptor));\\n\\n\\tfor (n = 0; n < num_descriptors; n++)\\n \\t\\tif (hdesc->desc[n].bDescriptorType == HID_DT_REPORT)\\n \\t\\t\\trsize = le16_to_cpu(hdesc->desc[n].wDescriptorLength);\\n \\n\\tif (!rsize || rsize > HID_MAX_DESCRIPTOR_SIZE) {\\n\\t\\tdbg_hid(\\"weird size of report descriptor (%u)\\\\n\\", rsize);\\n\\t\\treturn -EINVAL;\\n\\t}\\n\\n\\trdesc = kmalloc(rsize, GFP_KERNEL);\\n\\tif (!rdesc)\\n\\t\\treturn -ENOMEM;\\n\\n\\thid_set_idle(dev, interface->desc.bInterfaceNumber, 0, 0);\\n\\n\\tret = hid_get_class_descriptor(dev, interface->desc.bInterfaceNumber,\\n\\t\\t\\tHID_DT_REPORT, rdesc, rsize);\\n\\tif (ret < 0) {\\n\\t\\tdbg_hid(\\"reading report descriptor failed\\\\n\\");\\n\\t\\tkfree(rdesc);\\n\\t\\tgoto err;\\n\\t}\\n\\n\\tret = hid_parse_report(hid, rdesc, rsize);\\n\\tkfree(rdesc);\\n\\tif (ret) {\\n\\t\\tdbg_hid(\\"parsing report descriptor failed\\\\n\\");\\n\\t\\tgoto err;\\n\\t}\\n\\n\\thid->quirks |= quirks;\\n\\n\\treturn 0;\\nerr:\\n\\treturn ret;\\n}\\n"}, {"func_before": "MagickExport size_t GetQuantumExtent(const Image *image,\\n  const QuantumInfo *quantum_info,const QuantumType quantum_type)\\n{\\n  size_t\\n    extent,\\n    packet_size;\\n\\n  assert(quantum_info != (QuantumInfo *) NULL);\\n  assert(quantum_info->signature == MagickCoreSignature);\\n  packet_size=1;\\n  switch (quantum_type)\\n  {\\n    case GrayAlphaQuantum: packet_size=2; break;\\n    case IndexAlphaQuantum: packet_size=2; break;\\n    case RGBQuantum: packet_size=3; break;\\n    case BGRQuantum: packet_size=3; break;\\n    case RGBAQuantum: packet_size=4; break;\\n    case RGBOQuantum: packet_size=4; break;\\n     case BGRAQuantum: packet_size=4; break;\\n     case CMYKQuantum: packet_size=4; break;\\n     case CMYKAQuantum: packet_size=5; break;\\n     default: break;\\n   }\\n   extent=MagickMax(image->columns,image->rows);\\n  if (quantum_info->pack == MagickFalse)\\n    return((size_t) (packet_size*extent*((quantum_info->depth+7)/8)));\\n  return((size_t) ((packet_size*extent*quantum_info->depth+7)/8));\\n}\\n", "func_after": "MagickExport size_t GetQuantumExtent(const Image *image,\\n  const QuantumInfo *quantum_info,const QuantumType quantum_type)\\n{\\n  size_t\\n    extent,\\n    packet_size;\\n\\n  assert(quantum_info != (QuantumInfo *) NULL);\\n  assert(quantum_info->signature == MagickCoreSignature);\\n  packet_size=1;\\n  switch (quantum_type)\\n  {\\n    case GrayAlphaQuantum: packet_size=2; break;\\n    case IndexAlphaQuantum: packet_size=2; break;\\n    case RGBQuantum: packet_size=3; break;\\n    case BGRQuantum: packet_size=3; break;\\n    case RGBAQuantum: packet_size=4; break;\\n    case RGBOQuantum: packet_size=4; break;\\n     case BGRAQuantum: packet_size=4; break;\\n     case CMYKQuantum: packet_size=4; break;\\n     case CMYKAQuantum: packet_size=5; break;\\n    case CbYCrAQuantum: packet_size=4; break;\\n    case CbYCrQuantum: packet_size=3; break;\\n    case CbYCrYQuantum: packet_size=4; break;\\n     default: break;\\n   }\\n   extent=MagickMax(image->columns,image->rows);\\n  if (quantum_info->pack == MagickFalse)\\n    return((size_t) (packet_size*extent*((quantum_info->depth+7)/8)));\\n  return((size_t) ((packet_size*extent*quantum_info->depth+7)/8));\\n}\\n"}, {"func_before": "GF_Err urn_Read(GF_Box *s, GF_BitStream *bs)\\n{\\n\\tu32 i, to_read;\\n\\tchar *tmpName;\\n\\tGF_DataEntryURNBox *ptr = (GF_DataEntryURNBox *)s;\\n\\tif (! ptr->size ) return GF_OK;\\n\\n\\tto_read = (u32) ptr->size;\\n\\ttmpName = (char*)gf_malloc(sizeof(char) * to_read);\\n\\tif (!tmpName) return GF_OUT_OF_MEM;\\n\\tgf_bs_read_data(bs, tmpName, to_read);\\n \\n \\ti = 0;\\n\\twhile ( (tmpName[i] != 0) && (i < to_read) ) {\\n \\t\\ti++;\\n \\t}\\n\\tif (i == to_read) {\\n\\t\\tgf_free(tmpName);\\n\\t\\treturn GF_ISOM_INVALID_FILE;\\n\\t}\\n\\tif (i == to_read - 1) {\\n\\t\\tptr->nameURN = tmpName;\\n\\t\\tptr->location = NULL;\\n\\t\\treturn GF_OK;\\n\\t}\\n\\tptr->nameURN = (char*)gf_malloc(sizeof(char) * (i+1));\\n\\tif (!ptr->nameURN) {\\n\\t\\tgf_free(tmpName);\\n\\t\\treturn GF_OUT_OF_MEM;\\n\\t}\\n\\tptr->location = (char*)gf_malloc(sizeof(char) * (to_read - i - 1));\\n\\tif (!ptr->location) {\\n\\t\\tgf_free(tmpName);\\n\\t\\tgf_free(ptr->nameURN);\\n\\t\\tptr->nameURN = NULL;\\n\\t\\treturn GF_OUT_OF_MEM;\\n\\t}\\n\\tmemcpy(ptr->nameURN, tmpName, i + 1);\\n\\tmemcpy(ptr->location, tmpName + i + 1, (to_read - i - 1));\\n\\tgf_free(tmpName);\\n\\treturn GF_OK;\\n}\\n", "func_after": "GF_Err urn_Read(GF_Box *s, GF_BitStream *bs)\\n{\\n\\tu32 i, to_read;\\n\\tchar *tmpName;\\n\\tGF_DataEntryURNBox *ptr = (GF_DataEntryURNBox *)s;\\n\\tif (! ptr->size ) return GF_OK;\\n\\n\\tto_read = (u32) ptr->size;\\n\\ttmpName = (char*)gf_malloc(sizeof(char) * to_read);\\n\\tif (!tmpName) return GF_OUT_OF_MEM;\\n\\tgf_bs_read_data(bs, tmpName, to_read);\\n \\n \\ti = 0;\\n\\twhile ( (i < to_read) && (tmpName[i] != 0) ) {\\n \\t\\ti++;\\n \\t}\\n\\tif (i == to_read) {\\n\\t\\tgf_free(tmpName);\\n\\t\\treturn GF_ISOM_INVALID_FILE;\\n\\t}\\n\\tif (i == to_read - 1) {\\n\\t\\tptr->nameURN = tmpName;\\n\\t\\tptr->location = NULL;\\n\\t\\treturn GF_OK;\\n\\t}\\n\\tptr->nameURN = (char*)gf_malloc(sizeof(char) * (i+1));\\n\\tif (!ptr->nameURN) {\\n\\t\\tgf_free(tmpName);\\n\\t\\treturn GF_OUT_OF_MEM;\\n\\t}\\n\\tptr->location = (char*)gf_malloc(sizeof(char) * (to_read - i - 1));\\n\\tif (!ptr->location) {\\n\\t\\tgf_free(tmpName);\\n\\t\\tgf_free(ptr->nameURN);\\n\\t\\tptr->nameURN = NULL;\\n\\t\\treturn GF_OUT_OF_MEM;\\n\\t}\\n\\tmemcpy(ptr->nameURN, tmpName, i + 1);\\n\\tmemcpy(ptr->location, tmpName + i + 1, (to_read - i - 1));\\n\\tgf_free(tmpName);\\n\\treturn GF_OK;\\n}\\n"}], "CWE-189": [{"func_before": "asn1_get_bit_der (const unsigned char *der, int der_len,\\n \\t\\t  int *ret_len, unsigned char *str, int str_size,\\n \\t\\t  int *bit_len)\\n {\\n  int len_len, len_byte;\\n \\n   if (der_len <= 0)\\n     return ASN1_GENERIC_ERROR;\\n  len_byte = asn1_get_length_der (der, der_len, &len_len) - 1;\\n  if (len_byte < 0)\\n    return ASN1_DER_ERROR;\\n \\n   *ret_len = len_byte + len_len + 1;\\n   *bit_len = len_byte * 8 - der[len_len];\\n \\n   if (str_size >= len_byte)\\n     memcpy (str, der + len_len + 1, len_byte);\\n    }\\n", "func_after": "asn1_get_bit_der (const unsigned char *der, int der_len,\\n \\t\\t  int *ret_len, unsigned char *str, int str_size,\\n \\t\\t  int *bit_len)\\n {\\n  int len_len = 0, len_byte;\\n \\n   if (der_len <= 0)\\n     return ASN1_GENERIC_ERROR;\\n  len_byte = asn1_get_length_der (der, der_len, &len_len) - 1;\\n  if (len_byte < 0)\\n    return ASN1_DER_ERROR;\\n \\n   *ret_len = len_byte + len_len + 1;\\n   *bit_len = len_byte * 8 - der[len_len];\\n  \\n  if (*bit_len <= 0)\\n    return ASN1_DER_ERROR;\\n \\n   if (str_size >= len_byte)\\n     memcpy (str, der + len_len + 1, len_byte);\\n    }\\n"}, {"func_before": "makepol(QPRS_STATE *state)\\n{\\n\\tint32\\t\\tval = 0,\\n\\t\\t\\t\\ttype;\\n\\tint32\\t\\tlenval = 0;\\n\\tchar\\t   *strval = NULL;\\n\\tint32\\t\\tstack[STACKDEPTH];\\n \\tint32\\t\\tlenstack = 0;\\n \\tuint16\\t\\tflag = 0;\\n \\n \\twhile ((type = gettoken_query(state, &val, &lenval, &strval, &flag)) != END)\\n \\t{\\n \\t\\tswitch (type)\\n\\t\\t{\\n\\t\\t\\tcase VAL:\\n\\t\\t\\t\\tpushval_asis(state, VAL, strval, lenval, flag);\\n\\t\\t\\t\\twhile (lenstack && (stack[lenstack - 1] == (int32) \'&\' ||\\n\\t\\t\\t\\t\\t\\t\\t\\t\\tstack[lenstack - 1] == (int32) \'!\'))\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase OPR:\\n\\t\\t\\t\\tif (lenstack && val == (int32) \'|\')\\n\\t\\t\\t\\t\\tpushquery(state, OPR, val, 0, 0, 0);\\n\\t\\t\\t\\telse\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tif (lenstack == STACKDEPTH)\\n\\t\\t\\t\\t\\t\\t/* internal error */\\n\\t\\t\\t\\t\\t\\telog(ERROR, \\"stack too short\\");\\n\\t\\t\\t\\t\\tstack[lenstack] = val;\\n\\t\\t\\t\\t\\tlenstack++;\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase OPEN:\\n\\t\\t\\t\\tif (makepol(state) == ERR)\\n\\t\\t\\t\\t\\treturn ERR;\\n\\t\\t\\t\\twhile (lenstack && (stack[lenstack - 1] == (int32) \'&\' ||\\n\\t\\t\\t\\t\\t\\t\\t\\t\\tstack[lenstack - 1] == (int32) \'!\'))\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase CLOSE:\\n\\t\\t\\t\\twhile (lenstack)\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t};\\n\\t\\t\\t\\treturn END;\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase ERR:\\n\\t\\t\\tdefault:\\n\\t\\t\\t\\tereport(ERROR,\\n\\t\\t\\t\\t\\t\\t(errcode(ERRCODE_SYNTAX_ERROR),\\n\\t\\t\\t\\t\\t\\t errmsg(\\"syntax error\\")));\\n\\n\\t\\t\\t\\treturn ERR;\\n\\n\\t\\t}\\n\\t}\\n\\twhile (lenstack)\\n\\t{\\n\\t\\tlenstack--;\\n\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t};\\n\\treturn END;\\n}\\n", "func_after": "makepol(QPRS_STATE *state)\\n{\\n\\tint32\\t\\tval = 0,\\n\\t\\t\\t\\ttype;\\n\\tint32\\t\\tlenval = 0;\\n\\tchar\\t   *strval = NULL;\\n\\tint32\\t\\tstack[STACKDEPTH];\\n \\tint32\\t\\tlenstack = 0;\\n \\tuint16\\t\\tflag = 0;\\n \\n\\t/* since this function recurses, it could be driven to stack overflow */\\n\\tcheck_stack_depth();\\n\\n \\twhile ((type = gettoken_query(state, &val, &lenval, &strval, &flag)) != END)\\n \\t{\\n \\t\\tswitch (type)\\n\\t\\t{\\n\\t\\t\\tcase VAL:\\n\\t\\t\\t\\tpushval_asis(state, VAL, strval, lenval, flag);\\n\\t\\t\\t\\twhile (lenstack && (stack[lenstack - 1] == (int32) \'&\' ||\\n\\t\\t\\t\\t\\t\\t\\t\\t\\tstack[lenstack - 1] == (int32) \'!\'))\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase OPR:\\n\\t\\t\\t\\tif (lenstack && val == (int32) \'|\')\\n\\t\\t\\t\\t\\tpushquery(state, OPR, val, 0, 0, 0);\\n\\t\\t\\t\\telse\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tif (lenstack == STACKDEPTH)\\n\\t\\t\\t\\t\\t\\t/* internal error */\\n\\t\\t\\t\\t\\t\\telog(ERROR, \\"stack too short\\");\\n\\t\\t\\t\\t\\tstack[lenstack] = val;\\n\\t\\t\\t\\t\\tlenstack++;\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase OPEN:\\n\\t\\t\\t\\tif (makepol(state) == ERR)\\n\\t\\t\\t\\t\\treturn ERR;\\n\\t\\t\\t\\twhile (lenstack && (stack[lenstack - 1] == (int32) \'&\' ||\\n\\t\\t\\t\\t\\t\\t\\t\\t\\tstack[lenstack - 1] == (int32) \'!\'))\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t}\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase CLOSE:\\n\\t\\t\\t\\twhile (lenstack)\\n\\t\\t\\t\\t{\\n\\t\\t\\t\\t\\tlenstack--;\\n\\t\\t\\t\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t\\t\\t\\t};\\n\\t\\t\\t\\treturn END;\\n\\t\\t\\t\\tbreak;\\n\\t\\t\\tcase ERR:\\n\\t\\t\\tdefault:\\n\\t\\t\\t\\tereport(ERROR,\\n\\t\\t\\t\\t\\t\\t(errcode(ERRCODE_SYNTAX_ERROR),\\n\\t\\t\\t\\t\\t\\t errmsg(\\"syntax error\\")));\\n\\n\\t\\t\\t\\treturn ERR;\\n\\n\\t\\t}\\n\\t}\\n\\twhile (lenstack)\\n\\t{\\n\\t\\tlenstack--;\\n\\t\\tpushquery(state, OPR, stack[lenstack], 0, 0, 0);\\n\\t};\\n\\treturn END;\\n}\\n"}, {"func_before": "void CrosLibrary::TestApi::SetCryptohomeLibrary(\\n    CryptohomeLibrary* library, bool own) {\\n  library_->crypto_lib_.SetImpl(library, own);\\n}\\n", "func_after": "void CrosLibrary::TestApi::SetCryptohomeLibrary(\\n"}]}')
print('Fewshot pool:', {k: len(v) for k, v in FEWSHOT_DATA.items()})

In [ ]:
import re

def zero_shot_prompt(func_before, cwe_id):
    return (
        f'The following C/C++ function contains a {cwe_id} vulnerability.\n'
        f'Rewrite the function to fix the vulnerability.\n'
        f'Return ONLY the fixed function with no explanation or commentary.\n\n'
        f'Vulnerable function:\n```c\n{func_before}\n```\n\n'
        f'Fixed function:\n```c\n'
    )

def few_shot_prompt(func_before, cwe_id, fewshot_lookup):
    examples = fewshot_lookup.get(cwe_id, [])[:2]
    if not examples:
        return zero_shot_prompt(func_before, cwe_id)
    header = f'The following examples show how to fix {cwe_id} vulnerabilities in C/C++.\n\n'
    blocks = ''
    for i, ex in enumerate(examples, 1):
        blocks += (
            f'### Example {i}\n'
            f'Vulnerable:\n```c\n{ex["func_before"]}\n```\n'
            f'Fixed:\n```c\n{ex["func_after"]}\n```\n\n'
        )
    target = (
        f'### Now fix this function\n'
        f'Vulnerable:\n```c\n{func_before}\n```\n'
        f'Fixed:\n```c\n'
    )
    return header + blocks + target

def cot_prompt(func_before, cwe_id):
    return (
        f'The C/C++ function below contains a {cwe_id} vulnerability.\n'
        f'Identify where the vulnerability occurs, what causes it, '
        f'and what specific change is needed to fix it.\n'
        f'Then write the complete corrected function.\n\n'
        f'Vulnerable function:\n```c\n{func_before}\n```\n\n'
        f'Fixed function:\n```c\n'
    )

def extract_patch(generated_raw, prompt_type):
    match = re.search(r'^(.*?)(?:```|$)', generated_raw, re.DOTALL)
    if match:
        code = match.group(1).strip()
        if code:
            return code
    return generated_raw.strip()

print('Prompt builder loaded.')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading {MODEL_NAME} ...')
load_kwargs = dict(torch_dtype=torch.float16, device_map='auto')
if HF_TOKEN:
    load_kwargs['token'] = HF_TOKEN

tok_kwargs = dict(token=HF_TOKEN) if HF_TOKEN else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded on {next(model.parameters()).device}')
if device.type == 'cuda':
    for i in range(torch.cuda.device_count()):
        used  = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i} VRAM used: {used:.1f} / {total:.1f} GB')

In [ ]:
import pandas as pd, time

df = pd.read_csv(TEST_CSV_PATH, encoding='utf-8')
print(f'Test functions: {len(df)}')

prompt_builders = {
    'zero_shot': lambda func, cwe: zero_shot_prompt(func, cwe),
    'few_shot':  lambda func, cwe: few_shot_prompt(func, cwe, FEWSHOT_DATA),
    'cot':       lambda func, cwe: cot_prompt(func, cwe),
}

def load_done_set(path):
    done = set()
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    rec = json.loads(line)
                    done.add((rec['func_id'], rec['prompt_type'], rec['sample_idx']))
                except json.JSONDecodeError:
                    pass
    except FileNotFoundError:
        pass
    return done

def generate_one(prompt):
    tokenizer.truncation_side = 'left'
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    ).to(device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

done = load_done_set(OUTPUT_FILE)
total_calls = len(df) * len(prompt_builders) * NUM_SAMPLES
print(f'Total calls: {total_calls}  |  Already done: {len(done)}  |  Remaining: {total_calls - len(done)}')

t_start = time.time()
with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
    for i, row in df.iterrows():
        func_id, cwe_id = row['func_id'], row['cwe_id']
        func_before, func_after = row['func_before'], row['func_after']
        for prompt_type, builder in prompt_builders.items():
            prompt = builder(func_before, cwe_id)
            for sample_idx in range(NUM_SAMPLES):
                if (func_id, prompt_type, sample_idx) in done:
                    continue
                try:
                    raw   = generate_one(prompt)
                    patch = extract_patch(raw, prompt_type)
                    rec   = {
                        'model_alias': MODEL_ALIAS,
                        'func_id': func_id, 'cwe_id': cwe_id,
                        'prompt_type': prompt_type, 'sample_idx': sample_idx,
                        'patch': patch, 'generated_raw': raw,
                        'func_before': func_before, 'func_after': func_after,
                    }
                    out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
                    out_f.flush()
                    done.add((func_id, prompt_type, sample_idx))
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower():
                        torch.cuda.empty_cache()
                        print(f'OOM: {func_id}/{prompt_type}/{sample_idx} — skipping')
                    else:
                        raise
        elapsed = time.time() - t_start
        remaining = total_calls - len(done)
        rate = len(done) / elapsed if elapsed > 0 else 0
        eta  = remaining / rate / 60 if rate > 0 else 0
        print(f'[{i+1}/{len(df)}] {func_id} | done={len(done)} | '
              f'elapsed={elapsed/60:.1f}m | eta~{eta:.0f}m')

print(f'\nDone. {len(done)} records written to {OUTPUT_FILE}')

In [ ]:
from collections import Counter
counts, total = Counter(), 0
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            rec = json.loads(line)
            counts[rec['prompt_type']] += 1
            total += 1
        except json.JSONDecodeError:
            pass
print(f'Output : {OUTPUT_FILE}')
print(f'Total  : {total} records')
for pt, n in sorted(counts.items()):
    print(f'  {pt}: {n}')
print()
print('Next steps:')
print('  1. Click the Output tab (right panel) → download patches_raw_{MODEL_ALIAS}.jsonl')
print('  2. Copy it to your local results/ folder')
print(f'  3. Run: python validate_patches.py --input results/patches_raw_{MODEL_ALIAS}.jsonl')